# Polity Classification using Cliopatria

Use the [Cliopatria dataset](https://www.nature.com/articles/s41597-025-04516-9) — a GeoJSON of ~15K polygon
records covering ~1,600 polities from 3400 BCE to 2024 CE — to assign polities to randomly
sampled people via point-in-polygon lookup.

**Approach:**
1. Load 1M sample of randomly sampled people (same as `sampling_classification.ipynb`)
2. Load Cliopatria GeoJSON as a GeoDataFrame
3. For each Holocene person with birth_year ≥ -3400, find Cliopatria polygons where:
   - `FromYear <= birth_year <= ToYear`
   - The polygon contains the person's (lon, lat) point
4. Analyze coverage, top polities, and compare with the deterministic + LLM approach

In [1]:
import sys, os
sys.path.insert(0, '..')
os.chdir('..')

from person import sample_person

os.chdir('BiggestPolities')

import numpy as np
import dill
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from tqdm import tqdm
import matplotlib.pyplot as plt
from collections import Counter

## 1. Load sample

In [2]:
N = 10**6
PICKLE_FILE = f'../famous_person_sample_{N}.pkl'

if os.path.exists(PICKLE_FILE):
    print(f'Loading {N:,} people from {PICKLE_FILE}...')
    with open(PICKLE_FILE, 'rb') as f:
        people = dill.load(f)
else:
    print(f'Sampling {N:,} people...')
    people = [sample_person(light=True) for _ in tqdm(range(N), desc='Sampling')]
    with open(PICKLE_FILE, 'wb') as f:
        dill.dump(people, f)
    print(f'Saved to {PICKLE_FILE}')

print(f'Loaded {len(people):,} people')

Loading 1,000,000 people from ../famous_person_sample_1000000.pkl...
Loaded 1,000,000 people


## 2. Load Cliopatria

In [3]:
GEOJSON_PATH = 'cliopatria_polities_only.geojson'

print('Loading Cliopatria GeoJSON...')
clio_raw = gpd.read_file(GEOJSON_PATH)
print(f'Loaded {len(clio_raw):,} features')

# Split into polities and containers
clio = clio_raw[~clio_raw['Name'].str.startswith('(')].copy()
clio_containers = clio_raw[clio_raw['Name'].str.startswith('(')].copy()

print(f'Polities: {len(clio):,} features, {clio["Name"].nunique()} unique names')
print(f'Containers: {len(clio_containers):,} features, {clio_containers["Name"].nunique()} unique names')
print(f'Year range: {clio_raw["FromYear"].min()} to {clio_raw["ToYear"].max()}')

Loading Cliopatria GeoJSON...
Loaded 15,690 features
Polities: 12,987 features, 1522 unique names
Containers: 2,703 features, 96 unique names
Year range: -3400 to 2024


## 3. Build people GeoDataFrame

Filter to Holocene people with birth_year ≥ -3400 (Cliopatria's start date) and
create Point geometries from their (lon, lat) coordinates.

In [4]:
# HYDE grid → lat/lon conversion (from location.py)
HYDE_RES = 0.0833  # 360/4320 degrees

def row_col_to_latlon(row, col):
    lon = round(-180 + (col + 0.5) * HYDE_RES, 2)
    lat = round(90 - (row + 0.5) * HYDE_RES, 2)
    return lat, lon

records = []
for i, p in enumerate(people):
    if p.era == 'Paleolithic':
        continue
    if p.birth_year < -3400:
        continue
    loc = p.location
    lat, lon = row_col_to_latlon(p.row, p.col)
    records.append({
        'idx': i,
        'birth_year': p.birth_year,
        'country': loc.country,
        'lat': lat,
        'lon': lon,
    })

people_df = pd.DataFrame(records)
people_gdf = gpd.GeoDataFrame(
    people_df,
    geometry=gpd.points_from_xy(people_df['lon'], people_df['lat']),
    crs='EPSG:4326',
)

n_total = len(people)
n_paleo = sum(1 for p in people if p.era == 'Paleolithic')
n_pre_clio = sum(1 for p in people if p.era == 'Holocene' and p.birth_year < -3400)
n_eligible = len(people_gdf)

print(f'Total people:          {n_total:>10,}')
print(f'Paleolithic:           {n_paleo:>10,} (excluded — no coordinates)')
print(f'Holocene pre-3400 BCE: {n_pre_clio:>10,} (excluded — before Cliopatria coverage)')
print(f'Eligible for lookup:   {n_eligible:>10,} ({100*n_eligible/n_total:.1f}%)')

Total people:           1,000,000
Paleolithic:               44,434 (excluded — no coordinates)
Holocene pre-3400 BCE:     50,532 (excluded — before Cliopatria coverage)
Eligible for lookup:      905,034 (90.5%)


## 4. Spatial + temporal join

For each person, find matching Cliopatria polygons where the person's birth year falls
within the polygon's time range and the person's location falls within the polygon.

Processed in century-sized batches for efficiency. After a strict point-in-polygon pass,
a buffer pass (0.1°) catches near-misses from HYDE grid (0.083°) and Cliopatria polygon
resolution (~0.07°) errors.

We run this twice: once with **polities** (smallest = most specific), once with
**containers** (broadest umbrella like "(British Empire)").

In [5]:
people_gdf['century'] = (people_gdf['birth_year'] // 100) * 100
BUFFER_DEG = 0.1

def run_spatial_join(people_gdf, clio_df, label, resolve='smallest'):
    """Run spatial+temporal join with strict pass + buffer pass.
    resolve: 'smallest' or 'largest' area for deduplication."""
    centuries = sorted(people_gdf['century'].unique())
    
    # --- Pass 1: strict ---
    all_matches = []
    for cent in tqdm(centuries, desc=f'{label} strict'):
        cent_end = cent + 99
        p_cent = people_gdf[people_gdf['century'] == cent]
        if len(p_cent) == 0: continue
        c_mask = (clio_df['FromYear'] <= cent_end) & (clio_df['ToYear'] >= cent)
        c_cent = clio_df[c_mask]
        if len(c_cent) == 0: continue
        joined = gpd.sjoin(p_cent, c_cent, how='inner', predicate='within')
        if len(joined) == 0: continue
        year_mask = (joined['FromYear'] <= joined['birth_year']) & (joined['birth_year'] <= joined['ToYear'])
        matched = joined[year_mask][['idx', 'birth_year', 'country', 'lat', 'lon', 'Name', 'FromYear', 'ToYear', 'Area']].copy()
        all_matches.append(matched)
    
    strict_idx = set(pd.concat(all_matches)['idx']) if all_matches else set()
    print(f'  Strict: {len(strict_idx):,} people')
    
    # --- Pass 2: buffer ---
    unmatched = people_gdf[~people_gdf['idx'].isin(strict_idx)].copy()
    clio_buf = clio_df.copy()
    clio_buf['geometry'] = clio_buf.geometry.buffer(BUFFER_DEG)
    
    buffer_matches = []
    for cent in tqdm(sorted(unmatched['century'].unique()), desc=f'{label} buffer'):
        cent_end = cent + 99
        p_cent = unmatched[unmatched['century'] == cent]
        if len(p_cent) == 0: continue
        c_mask = (clio_buf['FromYear'] <= cent_end) & (clio_buf['ToYear'] >= cent)
        c_cent = clio_buf[c_mask]
        if len(c_cent) == 0: continue
        joined = gpd.sjoin(p_cent, c_cent, how='inner', predicate='within')
        if len(joined) == 0: continue
        year_mask = (joined['FromYear'] <= joined['birth_year']) & (joined['birth_year'] <= joined['ToYear'])
        matched = joined[year_mask][['idx', 'birth_year', 'country', 'lat', 'lon', 'Name', 'FromYear', 'ToYear', 'Area']].copy()
        buffer_matches.append(matched)
    
    buf_df = pd.concat(buffer_matches, ignore_index=True) if buffer_matches else pd.DataFrame()
    n_buf = buf_df['idx'].nunique() if len(buf_df) > 0 else 0
    print(f'  Buffer: {n_buf:,} additional people')
    
    # --- Combine + resolve ---
    matches = pd.concat(all_matches + buffer_matches, ignore_index=True) if (all_matches or buffer_matches) else pd.DataFrame()
    if len(matches) > 0:
        ascending = (resolve == 'smallest')
        best = matches.sort_values('Area', ascending=ascending).drop_duplicates(subset='idx', keep='first')
    else:
        best = matches
    
    print(f'  Total: {len(best):,} / {len(people_gdf):,} ({100*len(best)/len(people_gdf):.1f}%)')
    return best

# Run for polities (smallest = most specific)
print('=== Polities (smallest) ===')
best = run_spatial_join(people_gdf, clio, 'Polities', resolve='smallest')
matched_idx = set(best['idx'])

# Run for containers (largest = broadest umbrella)
print('\n=== Containers (largest) ===')
best_containers = run_spatial_join(people_gdf, clio_containers, 'Containers', resolve='largest')

=== Polities (smallest) ===


Polities strict: 100%|██████████████████████████| 55/55 [00:33<00:00,  1.62it/s]


  Strict: 590,863 people


Polities buffer: 100%|██████████████████████████| 55/55 [00:02<00:00, 22.91it/s]


  Buffer: 17,964 additional people
  Total: 608,827 / 905,034 (67.3%)

=== Containers (largest) ===


Containers strict: 100%|████████████████████████| 55/55 [00:21<00:00,  2.60it/s]


  Strict: 210,466 people


Containers buffer: 100%|████████████████████████| 55/55 [00:13<00:00,  4.09it/s]

  Buffer: 8,396 additional people
  Total: 218,862 / 905,034 (24.2%)


## 5. Top polities by sampled births

Two views: **polities** (most specific political unit) and **containers** (broad umbrellas
like "(British Empire)", "(Holy Roman Empire)").

In [6]:
# Get year ranges from Cliopatria for each polity (both polities and containers)
polity_years = clio_raw.groupby('Name').agg(start=('FromYear', 'min'), end=('ToYear', 'max'))

def fmt_year(y):
    return f'{-y} BCE' if y < 0 else f'{y} CE'

def print_ranking(best_df, title, n=60):
    counts = best_df['Name'].value_counts()
    print(f'\n{"="*110}')
    print(f'{title} — {len(best_df):,} people matched')
    print(f'{"="*110}')
    print(f'{"Rank":<5} {"Polity":<55} {"Years":<25} {"Count":>7} {"% of all":>14}')
    print('-' * 110)
    for i, (polity, count) in enumerate(counts.head(n).items(), 1):
        p = count / n_total
        se = np.sqrt(p * (1 - p) / n_total) * 100
        yrs = polity_years.loc[polity]
        yr_str = f'{fmt_year(yrs["start"])}–{fmt_year(yrs["end"])}'
        print(f'{i:<5} {polity:<55} {yr_str:<25} {count:>7,} {100*p:>6.3f} ± {se:.3f}%')
    print(f'\n... {len(counts)} total')
    return counts

polity_counts = print_ranking(best, 'POLITIES (most specific)')
container_counts = print_ranking(best_containers, 'CONTAINERS (broadest umbrella)')


POLITIES (most specific) — 608,827 people matched
Rank  Polity                                                  Years                       Count       % of all
--------------------------------------------------------------------------------------------------------------
1     Qing Dynasty                                            1645 CE–1911 CE            37,491  3.749 ± 0.019%
2     Republic of India                                       1947 CE–2024 CE            28,655  2.865 ± 0.017%
3     People's Republic of China                              1950 CE–2024 CE            24,812  2.481 ± 0.016%
4     British Raj                                             1859 CE–1947 CE            19,581  1.958 ± 0.014%
5     Mughal Empire                                           1497 CE–1858 CE            14,459  1.446 ± 0.012%
6     Ming Dynasty                                            1375 CE–1644 CE            13,595  1.359 ± 0.012%
7     Roman Empire                                     

## 6. Person-years analysis

**Polity shares**: weight each person by 1/CBR to convert birth-proportional sample into
population-proportional shares. Pre-1600 uses default CBR of 42 per 1000.

**Total person-years**: total births × mean lifespan from sample.

In [7]:
# Load country-specific CBR data
import dill as dill_cbr
with open('../Processed_Data/processed_p1600_data.pkl', 'rb') as f:
    country_data = dill_cbr.load(f)

DEFAULT_CBR = 42.0  # births per 1000 population — pre-1600 / Paleolithic default
CURRENT_YEAR = 2026

def get_cbr(country, year):
    """Get crude birth rate (per 1000) for a country-year pair."""
    if country is None or country not in country_data:
        return DEFAULT_CBR
    cd = country_data[country]
    if year < cd.years[0] or year > cd.years[-1]:
        return DEFAULT_CBR
    return float(cd.CBR_f(year))

# --- 1/CBR weights: convert birth-proportional sample to population-proportional shares ---
all_cbr = np.array([
    get_cbr(
        people[i].location.country if people[i].era == 'Holocene' else None,
        people[i].birth_year
    ) for i in range(n_total)
])
all_weights = 1.0 / all_cbr
all_weights_norm = all_weights / all_weights.sum()

# --- Absolute totals ---
TOTAL_BIRTHS = 65_715_396_203  # from unified_births.pkl
TOTAL_BIRTHS_BILLIONS = TOTAL_BIRTHS / 1e9

years_lived = np.array([
    (CURRENT_YEAR - people[i].birth_year) if people[i].age_at_death == 'alive'
    else people[i].age_at_death
    for i in range(n_total)
])
TOTAL_PERSON_YEARS = TOTAL_BIRTHS * np.mean(years_lived)
TOTAL_PY_BILLIONS = TOTAL_PERSON_YEARS / 1e9

print(f'Total births: {TOTAL_BIRTHS_BILLIONS:.1f} billion')
print(f'Mean lifespan: {np.mean(years_lived):.1f} years')
print(f'Total person-years: {TOTAL_PY_BILLIONS:.0f} billion')

Total births: 65.7 billion
Mean lifespan: 27.1 years
Total person-years: 1783 billion


In [8]:
# Person-years ranking for both polities and containers
def compute_py_ranking(best_df, title):
    polity_py = {}
    for _, row in best_df.iterrows():
        idx = int(row['idx'])
        w = all_weights_norm[idx]
        polity_py[row['Name']] = polity_py.get(row['Name'], 0) + w
    py_rows = sorted(polity_py.items(), key=lambda x: -x[1])
    
    print(f'\n{"="*110}')
    print(f'PERSON-YEARS: {title}')
    print(f'{"="*110}')
    print(f'{"Rank":<5} {"Polity":<55} {"Years":<25} {"% person-yrs":>12}')
    print('-' * 100)
    for i, (polity, w) in enumerate(py_rows[:50], 1):
        yrs = polity_years.loc[polity]
        yr_str = f'{fmt_year(yrs["start"])}–{fmt_year(yrs["end"])}'
        print(f'{i:<5} {polity:<55} {yr_str:<25} {100*w:>9.3f}%')
    print(f'\n... {len(polity_py)} total')
    return py_rows

py_polities = compute_py_ranking(best, 'Polities (most specific)')
py_containers = compute_py_ranking(best_containers, 'Containers (broadest umbrella)')


PERSON-YEARS: Polities (most specific)
Rank  Polity                                                  Years                     % person-yrs
----------------------------------------------------------------------------------------------------
1     People's Republic of China                              1950 CE–2024 CE               4.581%
2     Republic of India                                       1947 CE–2024 CE               3.787%
3     Qing Dynasty                                            1645 CE–1911 CE               3.614%
4     United States of America                                1776 CE–2024 CE               1.647%
5     British Raj                                             1859 CE–1947 CE               1.647%
6     Mughal Empire                                           1497 CE–1858 CE               1.293%
7     Ming Dynasty                                            1375 CE–1644 CE               1.220%
8     Roman Empire                                            31 

## 7. Blog post cleanup

Clean up the polity rankings for presentation:

1. **Merge** name variants that are the same continuous state
2. **Rename** confusingly named polities
3. **Exclude** non-polity buckets (HRE Minor States)
4. **Split** Republic of China into pre-1950 (mainland) and post-1950 (Taiwan)
5. **Super-polity groups**: broader lineages not captured by Cliopatria containers

In [9]:
# --- Merge table: same continuous state, different Cliopatria names ---
POLITY_MERGES = {
    # Same state, name change at a point in time
    'Eastern Roman Empire': 'Byzantine Empire',
    'Kuomintang': 'Republic of China (mainland)',
    'Empire of China': 'Republic of China (mainland)',  # Yuan Shikai 1916
    'Union of Myanmar': 'Myanmar',
    # Brazil: four names for the same country
    'Brazilian Republic': 'Brazil',
    'Federated Republic of Brazil': 'Brazil',
    'Republic of Brazil': 'Brazil',
    'Unitary State of Brazil': 'Brazil',
    # Germany: pre/post reunification
    'Federal Republic of Germany': 'West/Reunified Germany',
    'Federated Republic of Germany': 'West/Reunified Germany',
    # Egypt mislabelled as UAR
    'United Arab Republic': 'Egypt',
    # Denmark mislabelled as Denmark-Norway after 1814
    # (handled in clean_polity_name below)
    # Mali mislabelled as Mali Federation after 1960
    'Mali Federation': 'Mali',
    # Egypt: three names
    'Arab Republic of Egypt': 'Egypt',
    # Iraq: typo + split
    'Iragi Republic': 'Iraq',
    'Republic of Iraq': 'Iraq',
}

POLITY_RENAMES = {
    'British Empire': 'East India Company',
    'Bourbon Kingdom of France': 'Restoration France',
    'Napoleonic Batavia Republic': 'Dutch Gold Coast',
}

POLITY_EXCLUDE = {
    'Holy Roman Empire Minor States',
}

from math import log10, floor

def sig2(x):
    """Format to 2 significant figures, plain decimal with commas."""
    if x == 0:
        return '0'
    d = -floor(log10(abs(x))) + 1
    r = round(x, d)
    if d <= 0:
        return f'{int(r):,}'
    return f'{r:,.{d}f}'

def clean_polity_name(name, birth_year):
    if name in POLITY_EXCLUDE:
        return None
    if name in POLITY_MERGES:
        name = POLITY_MERGES[name]
    if name in POLITY_RENAMES:
        name = POLITY_RENAMES[name]
    # Split Republic of China: mainland vs Taiwan
    if name == 'Republic of China':
        return 'Republic of China (Taiwan)' if birth_year >= 1950 else 'Republic of China (mainland)'
    # Split Burma: pre-colonial (Konbaung) vs independent
    if name == 'Burma':
        return 'Burma (Konbaung)' if birth_year < 1948 else 'Myanmar'
    # Denmark-Norway → Denmark after 1814
    if name == 'Denmark-Norway' and birth_year >= 1814:
        return 'Denmark'
    # Kingdom of Italy: medieval (587–969) vs modern (1862–1946)
    if name == 'Kingdom of Italy':
        return 'Kingdom of Italy (Lombard/Carolingian)' if birth_year < 1000 else 'Kingdom of Italy (modern)'
    return name

best_clean = best.copy()
best_clean['CleanName'] = best_clean.apply(lambda r: clean_polity_name(r['Name'], r['birth_year']), axis=1)
best_clean = best_clean[best_clean['CleanName'].notna()]

clean_years = best_clean.groupby('CleanName').agg(
    min_year=('birth_year', 'min'),
    max_year=('birth_year', 'max'),
)

clean_py = {}
for _, row in best_clean.iterrows():
    name = row['CleanName']
    clean_py[name] = clean_py.get(name, 0) + all_weights_norm[int(row['idx'])]

clean_counts = best_clean['CleanName'].value_counts()

print(f'{"Rank":<5} {"Polity":<50} {"Years":<20} {"Births (M)":>10} {"Pers-yrs (B)":>12}')
print('-' * 95)
for i, (polity, count) in enumerate(clean_counts.head(300).items(), 1):
    p = count / n_total
    births_m = p * TOTAL_BIRTHS_BILLIONS * 1000
    py_b = clean_py.get(polity, 0) * TOTAL_PY_BILLIONS
    yrs = clean_years.loc[polity]
    yr_str = f'{fmt_year(int(yrs["min_year"]))}–{fmt_year(int(yrs["max_year"]))}'
    print(f'{i:<5} {polity:<50} {yr_str:<20} {sig2(births_m):>10} {sig2(py_b):>12}')

print(f'\n{len(clean_counts)} polities after cleanup')

Rank  Polity                                             Years                Births (M) Pers-yrs (B)
-----------------------------------------------------------------------------------------------
1     Qing Dynasty                                       1645 CE–1911 CE           2,500           64
2     Republic of India                                  1947 CE–2024 CE           1,900           68
3     People's Republic of China                         1950 CE–2024 CE           1,600           82
4     British Raj                                        1859 CE–1947 CE           1,300           29
5     Mughal Empire                                      1503 CE–1858 CE             950           23
6     Ming Dynasty                                       1375 CE–1644 CE             890           22
7     Roman Empire                                       31 BCE–394 CE               720           17
8     Russian Empire                                     1721 CE–1916 CE             630

In [12]:
# --- Same table sorted by person-years ---
clean_py_sorted = sorted(clean_py.items(), key=lambda x: -x[1])

print(f'{"Rank":<5} {"Polity":<45} {"Years":<20} {"Births (M)":>10} {"Pers-yrs (B)":>12}')
print('-' * 95)
for i, (polity, py_frac) in enumerate(clean_py_sorted[:300], 1):
    count = clean_counts.get(polity, 0)
    p = count / n_total
    births_m = p * TOTAL_BIRTHS_BILLIONS * 1000
    py_b = py_frac * TOTAL_PY_BILLIONS
    yrs = clean_years.loc[polity]
    yr_str = f'{fmt_year(int(yrs["min_year"]))}–{fmt_year(int(yrs["max_year"]))}'
    print(f'{i:<5} {polity:<45} {yr_str:<20} {(births_m):>10} {(py_b):>12}')

Rank  Polity                                        Years                Births (M) Pers-yrs (B)
-----------------------------------------------------------------------------------------------
1     People's Republic of China                    1950 CE–2024 CE      1630.530410588836 81.69413491280284
2     Republic of India                             1947 CE–2024 CE      1883.074678196965 67.53717882982409
3     Qing Dynasty                                  1645 CE–1911 CE      2463.7359190466727 64.4479228719206
4     United States of America                      1776 CE–2024 CE      593.2128815244811 29.37436248789465
5     British Raj                                   1859 CE–1947 CE      1286.7731730509431 29.370945813628815
6     Mughal Empire                                 1503 CE–1858 CE      950.178913699177 23.053084900496863
7     Ming Dynasty                                  1375 CE–1644 CE      893.4008113797848 21.751684763200558
8     Roman Empire                       

In [11]:
# --- Super-polity groups ---
# Mix of Cliopatria containers and manually defined groups, ranked together.

SUPER_POLITIES = {
    'Imperial China (canonical succession)': [
        'Qin Dynasty', 'Han Dynasty', 'Xin Dynasty',
        'Western Jin',
        'Sui Dynasty', 'Tang Dynasty',
        'Northern Song', 'Southern Song',
        'Yuan Dynasty', 'Ming Dynasty', 'Qing Dynasty',
    ],
    'Russia (Muscovy onward)': [
        'Grand Principality of Moscow', 'Tsardom of Russia', 'Russian Empire',
        'Russian Republic',
        'Union of Soviet Socialist Republics', 'Russian Federation',
    ],
    'Caliphate (Rashidun–Abbasid)': [
        'Rashidun Caliphate', 'Umayyad Caliphate', 'Abbasid Caliphate',
    ],
    'Roman state': [
        'Roman Republic', 'Roman Empire', 'Western Roman Empire',
        'Eastern Roman Empire', 'Byzantine Empire',
        'Nicaean Empire', 'Empire of Trebizond',
    ],
    'Genghisid domains': [
        'Mongol Empire',
        'Yuan Dynasty', 'Northern Yuan',
        'Golden Horde', 'Chagatai Khanate', 'Eastern Chagatai Khanate', 'Ilkhanate',
        'Crimean Khanate', 'Kazakh Khanate', 'Khanate of Kazan',
        'Astrakhan Khanate', 'Khanate of Sibir', 'Qasim Khanate',
        'Khanate of Bukhara', 'Khanate of Khiva', 'Mongol Khanate',
    ],
    'Habsburg domains': [
        'House of Habsburg', 'Habsburg Monarchy',
        'Austrian Empire', 'Austria-Hungary',
        ('Kingdom of Bohemia', 1526, 9999),
        ('Kingdom of Hungary', 1526, 1918),
        'Principality of Transylvania',
        ('Kingdom of Spain', 1516, 1700),
        ('Spanish Empire', 1516, 1700),
        'Iberian Union', 'Pro-Habsburg Spain',
    ],
    'Bourbon domains': [
        ('Kingdom of France', 1589, 1792),
        'Bourbon Kingdom of France',
        ('New France', 1589, 1792),
        ('French India', 1589, 1792),
        ('Kingdom of Spain', 1700, 9999),
        ('Spanish Empire', 1700, 1879),
        'Kingdom of the Two Sicilies',
        ('Kingdom of Naples (Napoleonic)', 1734, 1806),
        'Duchy of Parma and Piacenza',
    ],
    'Japan (under the Emperor)': [
        'Yamato', 'Kamakura Shogunate', 'Ashikaga Shogunate',
        'Tokugawa Shogunate', 'Empire of Japan', 'Japan',
    ],
    'Portuguese Empire': [
        'County of Portugal', 'Kingdom of Portugal',
        'Portuguese Republic', 'Portugal',
        'Portuguese Empire', 'Portuguese Africa', 'Portuguese Colonies',
        'Portuguese Ceylon',
        'Empire of Brazil', 'Brazilian Republic',
        'Federated Republic of Brazil', 'Republic of Brazil',
        'Unitary State of Brazil',
    ],
    'Dutch Empire': [
        'Dutch Republic', 'Batavian Republic',
        'Sovereign Principality of the United Netherlands',
        'Netherlands', 'Netherlands Antilles',
        'Dutch East Indies', 'Dutch Cape Coast', 'Dutch Ceylon',
        'New Netherland', 'Napoleonic Batavia Republic',
    ],
    'German Empire': [
        'German Empire', 'German Africa',
    ],
}

# Cliopatria containers to include alongside the manual groups
CONTAINER_GROUPS = [
    '(British Empire)',
    '(Delhi Sultanate)',
    '(Holy Roman Empire)',
    '(Spanish Empire)',
]

def count_super_polity(members, best_df):
    idx_set = set()
    for member in members:
        if isinstance(member, tuple):
            name, start, end = member
            start = start if start is not None else -99999
            end = end if end is not None else 99999
            mask = (best_df['Name'] == name) & (best_df['birth_year'] >= start) & (best_df['birth_year'] <= end)
        else:
            mask = best_df['Name'] == member
        idx_set.update(best_df[mask]['idx'].values)
    return best_df[best_df['idx'].isin(idx_set)]

# Compute births and person-years for all groups
group_results = []

for group_name, members in SUPER_POLITIES.items():
    group_best = count_super_polity(members, best)
    pct_births = len(group_best) / n_total
    births_m = pct_births * TOTAL_BIRTHS_BILLIONS * 1000
    py_frac = sum(all_weights_norm[int(idx)] for idx in group_best['idx'].values)
    py_b = py_frac * TOTAL_PY_BILLIONS
    group_results.append((group_name, births_m, py_b))

for container_name in CONTAINER_GROUPS:
    mask = best_containers['Name'] == container_name
    sub = best_containers[mask]
    pct_births = len(sub) / n_total
    births_m = pct_births * TOTAL_BIRTHS_BILLIONS * 1000
    py_frac = sum(all_weights_norm[int(idx)] for idx in sub['idx'].values)
    py_b = py_frac * TOTAL_PY_BILLIONS
    group_results.append((container_name, births_m, py_b))

# Sort by births
group_results.sort(key=lambda x: -x[1])

print(f'{"Rank":<5} {"Super-polity":<45} {"Births (M)":>10} {"Pers-yrs (B)":>12}')
print('-' * 75)
for i, (name, births_m, py_b) in enumerate(group_results, 1):
    print(f'{i:<5} {name:<45} {sig2(births_m):>10} {sig2(py_b):>12}')

Rank  Super-polity                                  Births (M) Pers-yrs (B)
---------------------------------------------------------------------------
1     Imperial China (canonical succession)              5,400          140
2     (British Empire)                                   2,400           61
3     Roman state                                        1,200           30
4     Russia (Muscovy onward)                            1,200           38
5     Japan (under the Emperor)                            790           28
6     Genghisid domains                                    710           17
7     Habsburg domains                                     570           14
8     (Delhi Sultanate)                                    470           11
9     Portuguese Empire                                    450           16
10    (Holy Roman Empire)                                  450           11
11    Bourbon domains                                      440           14
12    (Spani